# VocalCoach Colab Training

Two-stage training to solve the pitch/technique multi-task conflict:
- **Stage 1**: Train pitch+VAD only (no technique) for 50 epochs so the backbone establishes clean pitch representations
- **Stage 2**: Resume from stage 1 checkpoint and add the technique head for 50 more epochs

Run cells in order. Cells 1–4 are setup (run once per session). Then pick **one** of the experiment sections.

**Before starting**: upload `NanoPitch_data.zip` to your Google Drive root.
Zip created locally with:
```bash
cd ~/NanoPitch-MusicalAI
zip -j -1 -v NanoPitch_data.zip \
    data/clean.npz data/noise.npz data/test.npz \
    data/vocalset/technique_train.npz \
    data/vocalset/technique_test.npz
```

## Cell 1 — GPU check

In [ ]:
!nvidia-smi
import torch
print(f"PyTorch:        {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU:            {torch.cuda.get_device_name(0)}")
print(f"VRAM:           {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## Cell 2 — Mount Drive and extract data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import zipfile, os

os.makedirs('/content/data/vocalset', exist_ok=True)

print("Extracting NanoPitch_data.zip...")
with zipfile.ZipFile('/content/drive/MyDrive/NanoPitch_data.zip', 'r') as z:
    for name in ['clean.npz', 'noise.npz', 'test.npz']:
        print(f"  {name}...")
        z.extract(name, '/content/data/')
    for name in ['technique_train.npz', 'technique_test.npz']:
        print(f"  {name}...")
        z.extract(name, '/content/data/vocalset/')

print("\nExtraction complete.")
!ls -lh /content/data/
!ls -lh /content/data/vocalset/

## Cell 3 — Clone repo and install dependencies

In [ ]:
!git clone https://github.com/YOUR_USERNAME/NanoPitch-MusicalAI /content/NanoPitch-MusicalAI
%cd /content/NanoPitch-MusicalAI
!pip install -r requirements.txt --quiet
print("Setup complete.")

## Cell 4 — Verify data loads correctly

In [ ]:
import numpy as np

clean = np.load('/content/data/clean.npz')
print(f"clean.npz keys:    {list(clean.keys())}")
print(f"  mel shape:       {clean['mel'].shape}")
print(f"  lengths shape:   {clean['lengths'].shape}")

tech = np.load('/content/data/vocalset/technique_train.npz')
print(f"\ntechnique_train.npz keys: {list(tech.keys())}")
print(f"  clips:           {tech['lengths'].shape[0]}")

test = np.load('/content/data/test.npz')
print(f"\ntest.npz keys:     {list(test.keys())}")
print(f"  clips:           {test['clips'].shape[0]}")

---
## Experiment: Option A — Two-stage training

**Why two stages?** Joint training from epoch 1 causes the technique gradient (~10×
stronger than pitch at epoch 1) to immediately capture the backbone, collapsing
pitch posteriors. Stage 1 lets pitch establish clean representations first.

Run the architecture you want (TCN or Conformer), or both in parallel if you have two sessions.

Checkpoints write directly to Drive so a session disconnect loses at most one epoch.

### TCN — Stage 1: pitch + VAD only (50 epochs)

In [ ]:
!python vocalcoach/train.py \
    --arch tcn --causal false \
    --data-dir /content/data \
    --output-dir /content/drive/MyDrive/NanoPitch-runs/tcn_stage1_pitchonly \
    --epochs 50 --batch-size 32 \
    --w-pitch 2 --w-vad 0.05 \
    --augment noise_specaug \
    --patience 0

### TCN — Stage 2: resume + technique head (epochs 51–100)

Run this after Stage 1 completes. Can be a new session — data and repo setup (cells 1–3) must be re-run, but the checkpoint is already on Drive.

In [ ]:
!python vocalcoach/train.py \
    --arch tcn --causal false \
    --data-dir /content/data \
    --technique-dirs /content/data/vocalset \
    --output-dir /content/drive/MyDrive/NanoPitch-runs/tcn_stage2_technique \
    --epochs 100 --batch-size 32 \
    --w-pitch 2 --w-vad 0.05 --w-technique 2 \
    --augment noise_specaug \
    --technique-pos-weights 2.9 4.2 1.0 4.0 1.9 \
    --patience 0 \
    --resume /content/drive/MyDrive/NanoPitch-runs/tcn_stage1_pitchonly/checkpoints/best_loss.pth

### Conformer — Stage 1: pitch + VAD only (50 epochs)

In [ ]:
!python vocalcoach/train.py \
    --arch conformer --causal false \
    --data-dir /content/data \
    --output-dir /content/drive/MyDrive/NanoPitch-runs/conformer_stage1_pitchonly \
    --epochs 50 --batch-size 32 \
    --w-pitch 2 --w-vad 0.05 \
    --augment noise_specaug \
    --patience 0

### Conformer — Stage 2: resume + technique head (epochs 51–100)

In [ ]:
!python vocalcoach/train.py \
    --arch conformer --causal false \
    --data-dir /content/data \
    --technique-dirs /content/data/vocalset \
    --output-dir /content/drive/MyDrive/NanoPitch-runs/conformer_stage2_technique \
    --epochs 100 --batch-size 32 \
    --w-pitch 2 --w-vad 0.05 --w-technique 2 \
    --augment noise_specaug \
    --technique-pos-weights 2.9 4.2 1.0 4.0 1.9 \
    --patience 0 \
    --resume /content/drive/MyDrive/NanoPitch-runs/conformer_stage1_pitchonly/checkpoints/best_loss.pth

---
## Evaluate a completed run

Copy the run directory from Drive to local disk first (faster I/O than reading from Drive directly).

In [ ]:
RUN_NAME = "tcn_stage2_technique"  # change to the run you want to evaluate

import shutil
shutil.copytree(
    f'/content/drive/MyDrive/NanoPitch-runs/{RUN_NAME}',
    f'/content/runs/{RUN_NAME}',
    dirs_exist_ok=True
)

!python vocalcoach/evaluate.py \
    --checkpoint /content/runs/{RUN_NAME}/checkpoints/best_metric.pth \
    --data-dir /content/data \
    --technique-dir /content/data/vocalset